In [1]:
from semantic_router.encoders import HuggingFaceEncoder

encoder=HuggingFaceEncoder(model_name="LazarusNLP/all-indo-e5-small-v4")

c:\Users\Ammar\miniconda3\envs\scraping\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-14 11:52:54 - numexpr.utils - INFO - utils.py:164 - _init_num_threads() - NumExpr defaulting to 16 threads.


In [2]:
from pathlib import Path
import json
from langchain_core.documents import Document

output_dir = Path("semantic_chunker_results")
semantic_source_candidates = [
    output_dir / "semantic_only_chunks.jsonl",
    output_dir / "semantic_chunks.jsonl",
]
semantic_source_path = next((path for path in semantic_source_candidates if path.exists()), None)

if semantic_source_path is None:
    raise FileNotFoundError(
        f"File semantic belum ditemukan. Coba jalankan cell penyimpanan dulu di folder: {output_dir}"
    )

chunks = []
with open(semantic_source_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        record = json.loads(line)
        page_content = str(record.get("page_content", "")).strip()
        if not page_content:
            continue

        metadata = dict(record.get("metadata") or {})
        metadata.setdefault("chunker", "statistical_chunker")
        chunks.append(Document(page_content=page_content, metadata=metadata))

print("Jumlah semantic chunks:", len(chunks))
print(f"Semantic chunks loaded from: {semantic_source_path}")

Jumlah semantic chunks: 73785
Semantic chunks loaded from: semantic_chunker_results\semantic_chunks.jsonl


In [3]:
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings


class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name: str = "LazarusNLP/all-indo-e5-small-v4"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        passages = [f"passage: {text.strip()}" for text in texts]
        return self.model.encode(
            passages,
            batch_size=32,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        ).tolist()

    def embed_query(self, text):
        query = f"query: {text.strip()}"
        return self.model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True,
        )[0].tolist()


embeddings_model = SentenceTransformerEmbeddings()
sample_vector = embeddings_model.embed_query("uji embedding")
print("Model embedding:", embeddings_model.model_name)
print("Dimensi vektor:", len(sample_vector))

2026-04-14 11:52:58 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:219 - __init__() - Use pytorch device_name: cpu
2026-04-14 11:52:58 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - __init__() - Load pretrained SentenceTransformer: LazarusNLP/all-indo-e5-small-v4
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.89it/s]

Model embedding: LazarusNLP/all-indo-e5-small-v4
Dimensi vektor: 384


In [4]:
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings


class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name: str = "LazarusNLP/all-indo-e5-small-v4"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        passages = [f"passage: {text.strip()}" for text in texts]
        return self.model.encode(
            passages,
            batch_size=32,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        ).tolist()

    def embed_query(self, text):
        query = f"query: {text.strip()}"
        return self.model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True,
        )[0].tolist()


embeddings_model = SentenceTransformerEmbeddings()
sample_vector = embeddings_model.embed_query("uji embedding")
print("Model embedding:", embeddings_model.model_name)
print("Dimensi vektor:", len(sample_vector))

2026-04-14 11:53:04 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:219 - __init__() - Use pytorch device_name: cpu
2026-04-14 11:53:04 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - __init__() - Load pretrained SentenceTransformer: LazarusNLP/all-indo-e5-small-v4
Batches: 100%|██████████| 1/1 [00:00<00:00, 29.13it/s]

Model embedding: LazarusNLP/all-indo-e5-small-v4
Dimensi vektor: 384


In [5]:
from pathlib import Path
import json

output_dir = Path("semantic_chunker_results")
output_dir.mkdir(parents=True, exist_ok=True)
chunks_md_dir = output_dir / "chunks_md"
chunks_md_dir.mkdir(parents=True, exist_ok=True)

if "chunks" in globals():
    chunk_records = [
        {
            "page_content": doc.page_content,
            "metadata": doc.metadata,
        }
        for doc in chunks
    ]
    chunks_path = output_dir / "semantic_chunks.jsonl"
    with open(chunks_path, "w", encoding="utf-8") as f:
        for record in chunk_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Semantic chunks saved to: {chunks_path}")

    # Export juga ke Markdown + YAML frontmatter (satu file per chunk)
    try:
        to_md = document_to_markdown_frontmatter  # dari Cell 1
    except NameError:
        # Fallback kalau Cell 1 belum dijalankan di kernel saat ini
        def _metadata_to_frontmatter_local(metadata: dict) -> str:
            meta = dict(metadata or {})
            lines = ["---"]
            for key, value in meta.items():
                if value is None:
                    rendered = "null"
                elif isinstance(value, bool):
                    rendered = "true" if value else "false"
                elif isinstance(value, (int, float)):
                    rendered = str(value)
                else:
                    rendered = json.dumps(str(value), ensure_ascii=False)
                lines.append(f"{key}: {rendered}")
            lines.append("---")
            return "\n".join(lines)

        def to_md(doc):
            fm = _metadata_to_frontmatter_local(getattr(doc, "metadata", {}) or {})
            body = (getattr(doc, "page_content", "") or "").strip()
            return f"{fm}\n\n{body}\n"

    for i, doc in enumerate(chunks, start=1):
        md_path = chunks_md_dir / f"chunk_{i:06d}.md"
        with open(md_path, "w", encoding="utf-8") as md_f:
            md_f.write(to_md(doc))
    print(f"Markdown+frontmatter chunks saved to: {chunks_md_dir}")

metrics_source = Path("rag_evaluation_metrics.png")
metrics_target = output_dir / metrics_source.name
if metrics_source.exists():
    metrics_source.replace(metrics_target)
elif "fig" in globals():
    fig.savefig(metrics_target, dpi=150, bbox_inches="tight")

report_source = Path("rag_evaluation_report.json")
report_target = output_dir / report_source.name
if report_source.exists():
    report_source.replace(report_target)
elif "evaluation_report" in globals():
    with open(report_target, "w", encoding="utf-8") as f:
        json.dump(evaluation_report, f, indent=2, ensure_ascii=False)


Semantic chunks saved to: semantic_chunker_results\semantic_chunks.jsonl
Markdown+frontmatter chunks saved to: semantic_chunker_results\chunks_md


In [10]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import Qdrant
from langchain_core.documents import Document

# ----------------------------
# Config
# ----------------------------
# Qdrant server default REST port: 6333
qdrant_host = "localhost"
qdrant_port = 6333

collection_name = "pdf_rag_collection_v2"
batch_size = 1000
force_reindex = True  # True = hapus & bangun ulang collection

# ----------------------------
# Prepare data (chunks -> texts + metadatas)
# ----------------------------
normalized_chunks: list[Document] = []
for item in chunks:
    if isinstance(item, Document):
        normalized_chunks.append(item)
        continue

    if isinstance(item, str) and item.strip():
        normalized_chunks.append(
            Document(page_content=item.strip(), metadata={"chunker": "unknown"})
        )
        continue

    if isinstance(item, list):
        for sub_item in item:
            if isinstance(sub_item, Document):
                normalized_chunks.append(sub_item)
            elif isinstance(sub_item, str) and sub_item.strip():
                normalized_chunks.append(
                    Document(
                        page_content=sub_item.strip(), metadata={"chunker": "unknown"}
                    )
                )

texts = [doc.page_content for doc in normalized_chunks]
metadatas = [doc.metadata for doc in normalized_chunks]

# ----------------------------
# Client init (reuse if exists)
# ----------------------------
if "client" in globals() and isinstance(client, QdrantClient):
    pass
else:
    client = QdrantClient(host=qdrant_host, port=qdrant_port)

# ----------------------------
# Compatibility adapter
# ----------------------------
# qdrant-client versi baru mengganti .search() -> .query_points().
# Beberapa versi langchain_qdrant masih memanggil client.search(...).
def ensure_search_compat(qdrant_client: QdrantClient) -> None:
    if hasattr(qdrant_client, "search") or (not hasattr(qdrant_client, "query_points")):
        return

    def _search_adapter(
        collection_name,
        query_vector,
        query_filter=None,
        search_params=None,
        limit=10,
        offset=0,
        with_payload=True,
        with_vectors=False,
        score_threshold=None,
        consistency=None,
        **kwargs,
    ):
        using = None
        query = query_vector
        if (
            isinstance(query_vector, tuple)
            and len(query_vector) == 2
            and isinstance(query_vector[0], str)
        ):
            using, query = query_vector

        response = qdrant_client.query_points(
            collection_name=collection_name,
            query=query,
            using=using,
            query_filter=query_filter,
            search_params=search_params,
            limit=limit,
            offset=offset,
            with_payload=with_payload,
            with_vectors=with_vectors,
            score_threshold=score_threshold,
            consistency=consistency,
            **kwargs,
        )
        return getattr(response, "points", response)

    qdrant_client.search = _search_adapter  # type: ignore[attr-defined]

ensure_search_compat(client)

# ----------------------------
# Ensure collection exists
# ----------------------------
vector_size = len(sample_vector) if "sample_vector" in globals() else len(
    embeddings_model.embed_query("cek dimensi")
 )
vectors_config = VectorParams(size=vector_size, distance=Distance.COSINE)

created_or_recreated = False
if force_reindex:
    # Drop collection kalau ada, lalu buat ulang (menghindari recreate_collection yang deprecated)
    try:
        exists = client.collection_exists(collection_name)
    except Exception:
        try:
            client.get_collection(collection_name)
            exists = True
        except Exception:
            exists = False

    if exists:
        try:
            client.delete_collection(collection_name)
        except Exception:
            pass

    client.create_collection(collection_name=collection_name, vectors_config=vectors_config)
    created_or_recreated = True
else:
    try:
        exists = client.collection_exists(collection_name)
    except Exception:
        try:
            client.get_collection(collection_name)
            exists = True
        except Exception:
            exists = False

    if not exists:
        client.create_collection(collection_name=collection_name, vectors_config=vectors_config)
        created_or_recreated = True

# ----------------------------
# Vectorstore + indexing
# ----------------------------
vectorstore = Qdrant(
    client=client,
    collection_name=collection_name,
    embeddings=embeddings_model,
)

should_index = created_or_recreated
if should_index:
    for start in range(0, len(texts), batch_size):
        end = start + batch_size
        vectorstore.add_texts(
            texts=texts[start:end],
            metadatas=metadatas[start:end],
        )

# ----------------------------
# Retriever + summary
# ----------------------------
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20, "lambda_mult": 0.5},
)

try:
    collection_count = int(client.count(collection_name, exact=True).count)
except Exception:
    collection_info = client.get_collection(collection_name)
    collection_count = int(getattr(collection_info, "points_count", 0) or 0)

print("Qdrant:", f"{qdrant_host}:{qdrant_port}")
print("Mode:", "REINDEX" if should_index else "REUSE DB")
print("Collection:", collection_name)
print("Jumlah data di collection:", collection_count)
print("Batch size:", batch_size)

C:\Users\Ammar\AppData\Local\Temp\ipykernel_35232\3273259254.py:150: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use `QdrantVectorStore` instead.
  vectorstore = Qdrant(
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.61it/s]

Qdrant: localhost:6333
Mode: REINDEX
Collection: pdf_rag_collection_v2
Jumlah data di collection: 73785
Batch size: 1000


In [11]:
question = "bagaimana mengungkap fakta alam secara objektif?"
results = vectorstore.similarity_search_with_score(question, k=5)

print("Pertanyaan:", question)
print()
for index, (doc, score) in enumerate(results, start=1):
    print(f"Chunk {index} | score={score:.4f} | metadata={doc.metadata}")
    print(doc.page_content[:500])
    print("-" * 100)

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.72it/s]


Pertanyaan: bagaimana mengungkap fakta alam secara objektif?

Chunk 1 | score=0.5217 | metadata={'producer': 'GPL Ghostscript 9.05', 'creator': '', 'creationdate': '2024-03-22T21:03:54+07:00', 'source': 'Daftar Buku Pegangan Guru\\Salinan Bahasa_Indonesia_BS_KLS_X_Rev.pdf', 'file_path': 'Daftar Buku Pegangan Guru\\Salinan Bahasa_Indonesia_BS_KLS_X_Rev.pdf', 'total_pages': 320, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-03-22T21:03:54+07:00', 'trapped': '', 'modDate': "D:20240322210354+07'00'", 'creationDate': "D:20240322210354+07'00'", 'page': 38, 'source_file': 'Salinan Bahasa_Indonesia_BS_KLS_X_Rev.pdf', 'source_path': 'Daftar Buku Pegangan Guru\\Salinan Bahasa_Indonesia_BS_KLS_X_Rev.pdf', 'chunker': 'statistical_chunker', '_id': 'bde7b854-68c5-4562-b6fa-bc925b10c29e', '_collection_name': 'pdf_rag_collection_v2'}
splits=['Bab I | Mengungkap Fakta Alam secara Objektif'] is_triggered=True triggered_score=np.float64(0.1036627652038448

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    reasoning=False,
    num_predict=512,
    num_ctx=4096,
    seed=42,
    keep_alive="30m",
)

system_prompt = """
Jawab hanya berdasarkan konteks yang diberikan.
Jika informasi di konteks tidak cukup, katakan bahwa jawabannya belum cukup didukung konteks.
Rangkum jawaban secara singkat, jelas, dan dalam bahasa Indonesia.

Konteks:
{context}
""".strip()

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Pertanyaan: {input}"),
    ]
)

question = "Jelaskan apa itu teks biografi?"
docs = retriever.invoke(question)
context_blocks = []
for index, doc in enumerate(docs, start=1):
    source = doc.metadata.get("source_file", doc.metadata.get("source", "unknown"))
    page = doc.metadata.get("page", "unknown")
    context_blocks.append(
        f"[Chunk {index} | source={source} | page={page}]\n{doc.page_content}"
    )
context = "\n\n".join(context_blocks)

messages = prompt.format_messages(input=question, context=context)
response = llm.invoke(messages)

answer = (response.content or "").strip()
if not answer:
    answer = "Model belum mengembalikan jawaban teks. Coba jalankan ulang sekali lagi atau ganti model Ollama lain (mis. llama3.1:8b)."

print("Pertanyaan:", question)
print("\nKonteks terambil:")
for index, doc in enumerate(docs, start=1):
    source = doc.metadata.get("source_file", doc.metadata.get("source", "unknown"))
    page = doc.metadata.get("page", "unknown")
    print(f"- Chunk {index} | source={source} | page={page}")

print("\nJawaban:")
print(answer)

In [ ]:
import json
import numpy as np
import pandas as pd
from datetime import datetime

eval_questions = [
    "Jelaskan metode pembelajaran yang efektif untuk mengajar di kelas",
    "Apa itu kurikulum dan bagaimana implementasinya?",
]

retrieval_scores_all = []
source_diversity = []
generation_results = []

for question in eval_questions:
    # Retrieval
    scored_docs = vectorstore.similarity_search_with_score(question, k=5)
    docs = [doc for doc, _ in scored_docs]
    scores = [score for _, score in scored_docs]
    retrieval_scores_all.extend(scores)
    source_diversity.append(len(set(d.metadata.get("source_file", "unknown") for d in docs)))

    # Generation
    context = "\n\n".join([d.page_content for d in docs])
    messages = prompt.format_messages(input=question, context=context)
    response = llm.invoke(messages)
    answer = (response.content or "").strip() or "[No response]"

    generation_results.append({
        "question": question,
        "answer": answer,
        "word_count": len(answer.split()),
        "context_used": len(docs),
        "avg_context_length": float(np.mean([len(d.page_content.split()) for d in docs])) if docs else 0.0,
    })

metrics = {
    "mean_score": float(np.mean(retrieval_scores_all)) if retrieval_scores_all else 0.0,
    "std_score": float(np.std(retrieval_scores_all)) if retrieval_scores_all else 0.0,
    "min_score": float(np.min(retrieval_scores_all)) if retrieval_scores_all else 0.0,
    "max_score": float(np.max(retrieval_scores_all)) if retrieval_scores_all else 0.0,
    "avg_source_diversity": float(np.mean(source_diversity)) if source_diversity else 0.0,
}

gen_stats = {
    "avg_word_count": float(np.mean([r["word_count"] for r in generation_results])) if generation_results else 0.0,
    "avg_context_used": float(np.mean([r["context_used"] for r in generation_results])) if generation_results else 0.0,
}

high_relevance_count = sum(1 for s in retrieval_scores_all if s >= 0.7)
high_relevance_pct = (high_relevance_count / len(retrieval_scores_all) * 100) if retrieval_scores_all else 0.0

if high_relevance_pct >= 70 and 40 <= gen_stats["avg_word_count"] <= 220:
    system_status = "BAIK"
elif high_relevance_pct >= 55:
    system_status = "CUKUP"
else:
    system_status = "PERLU OPTIMASI"

print(f"Mean retrieval score : {metrics['mean_score']:.4f}")
print(f"High relevance@5     : {high_relevance_pct:.1f}%")
print(f"Avg answer length    : {gen_stats['avg_word_count']:.1f} kata")
print(f"Status sistem        : {system_status}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RAG System Evaluation Metrics', fontsize=16, fontweight='bold')


ax1 = axes[0, 0]
ax1.hist(retrieval_scores_all, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
ax1.axvline(metrics['mean_score'], color='red', linestyle='--', linewidth=2, label=f"Mean: {metrics['mean_score']:.3f}")
ax1.set_xlabel('Relevance Score')
ax1.set_ylabel('Frequency')
ax1.set_title('Retrieval Score Distribution')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[0, 1]
relevance_counts = {
    'High (≥0.7)': sum(1 for s in retrieval_scores_all if s >= 0.7),
    'Medium (0.5-0.7)': sum(1 for s in retrieval_scores_all if 0.5 <= s < 0.7),
    'Low (<0.5)': sum(1 for s in retrieval_scores_all if s < 0.5),
}
colors = ['#2ecc71', '#f39c12', '#e74c3c']
ax2.bar(relevance_counts.keys(), relevance_counts.values(), color=colors, edgecolor='black', alpha=0.7)
ax2.set_ylabel('Count')
ax2.set_title('Relevance Tier Distribution')
for i, (k, v) in enumerate(relevance_counts.items()):
    ax2.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

ax3 = axes[1, 0]
q_labels = [f"Q{i+1}" for i in range(len(generation_results))]
word_counts = [r['word_count'] for r in generation_results]
ax3.bar(q_labels, word_counts, color='coral', edgecolor='black', alpha=0.7)
ax3.set_ylabel('Word Count')
ax3.set_title('Answer Length by Question')
ax3.axhline(gen_stats['avg_word_count'], color='red', linestyle='--', label=f"Avg: {gen_stats['avg_word_count']:.0f}")
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Source Diversity
ax4 = axes[1, 1]
ax4.bar(q_labels, [r['avg_context_length'] for r in generation_results], 
        color='lightgreen', edgecolor='black', alpha=0.7)
ax4.set_ylabel('Avg Words per Context')
ax4.set_title('Average Context Length by Question')
ax4.grid(axis='y', alpha=0.3)




# Multimodal processing (partition_pdf + Moondream)
Bagian ini memproses semua PDF di folder **Daftar Buku Pegangan Guru** dengan `unstructured.partition.pdf.partition_pdf`, mengekstrak **Image** dan **Table** sebagai file gambar, lalu meringkasnya memakai model **moondream2**.

In [ ]:
# Jika belum ada, install dependensi untuk partition_pdf (Unstructured).
# Catatan Windows: untuk strategi hi_res + ekstraksi gambar, kadang butuh Poppler (pdf2image).
# Kalau error Poppler, lihat pesan error di output cell ini (solusinya biasanya install Poppler + set PATH).
%pip -q install "unstructured[pdf]"
%pip -q install "unstructured-inference"
%pip -q install "pillow"
%pip -q install "python-magic-bin"
%pip -q install "opencv-python"
%pip -q install "pytesseract"
%pip -q install "nltk"
%pip -q install "transformers" "accelerate" "sentencepiece"
import nltk
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

In [ ]:
from pathlib import Path
import json
import re
from typing import Any

from unstructured.partition.pdf import partition_pdf

# ----------------------------
# Input/Output paths
# ----------------------------
SOURCE_DIR = Path("Daftar Buku Pegangan Guru")
OUTPUT_ROOT = Path("multimodal_partition_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

pdf_paths = sorted(SOURCE_DIR.rglob("*.pdf"))
print(f"PDF ditemukan: {len(pdf_paths)}")
for p in pdf_paths[:10]:
    print("-", p)

def _safe_stem(path: Path) -> str:
    stem = path.stem
    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", stem).strip("_")
    return stem or "document"

def _serialize_metadata(meta: Any) -> dict:
    if meta is None:
        return {}
    if isinstance(meta, dict):
        return {k: _serialize_metadata(v) for k, v in meta.items()}
    # unstructured metadata objects often have .to_dict()
    if hasattr(meta, "to_dict"):
        try:
            return meta.to_dict()  # type: ignore[attr-defined]
        except Exception:
            pass
    # fallback: best-effort string
    return {"value": str(meta)}

def _element_to_record(el: Any) -> dict:
    el_type = getattr(el, "category", None) or getattr(el, "__class__", type("X", (), {})).__name__
    text = getattr(el, "text", "") or ""
    meta = getattr(el, "metadata", None)
    meta_dict = _serialize_metadata(meta)
    # Common keys for image paths depend on Unstructured version
    image_path = None
    for key in ("image_path", "image", "image_filename", "file_path"):
        if isinstance(meta_dict, dict) and key in meta_dict:
            image_path = meta_dict.get(key)
            break
    return {
        "type": str(el_type),
        "text": text,
        "metadata": meta_dict,
        "image_path": image_path,
    }

def partition_folder_pdfs(
    pdfs: list[Path],
    output_root: Path,
    strategy: str = "hi_res",
    infer_table_structure: bool = True,
) -> dict[str, Any]:
    """Partition each PDF; save element records + extracted images; return summary index."""
    index: dict[str, Any] = {"pdfs": []}
    for pdf_path in pdfs:
        pdf_id = _safe_stem(pdf_path)
        out_dir = output_root / pdf_id
        images_dir = out_dir / "images"
        out_dir.mkdir(parents=True, exist_ok=True)
        images_dir.mkdir(parents=True, exist_ok=True)

        print("\n=== Partition:", pdf_path)
        print("Output:", out_dir)

        try:
            elements = partition_pdf(
                filename=str(pdf_path),
                strategy=strategy,
                infer_table_structure=infer_table_structure,
                extract_images_in_pdf=True,
                extract_image_block_types=["Image", "Table"],
                extract_image_block_to_payload=False,
                image_output_dir_path=str(images_dir),
            )
            error = None
        except Exception as e:
            # Fallback: text-only partitioning (still useful if poppler is missing)
            print("[WARN] partition_pdf hi_res + images gagal, fallback ke fast/text-only.")
            print("       Error:", repr(e))
            try:
                elements = partition_pdf(
                    filename=str(pdf_path),
                    strategy="fast",
                    infer_table_structure=False,
                    extract_images_in_pdf=False,
                )
                error = repr(e)
            except Exception as e2:
                print("[ERROR] partition_pdf fallback juga gagal:", repr(e2))
                index["pdfs"].append({
                    "pdf": str(pdf_path),
                    "pdf_id": pdf_id,
                    "out_dir": str(out_dir),
                    "error": repr(e2),
                })
                continue

        # Save element records
        records = [_element_to_record(el) for el in elements]
        elements_path = out_dir / "elements.jsonl"
        with open(elements_path, "w", encoding="utf-8") as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        # List extracted images (tables and images rendered as image files)
        extracted_images = sorted([p for p in images_dir.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}])
        print(f"Elements: {len(records)} | Extracted images: {len(extracted_images)}")

        index["pdfs"].append({
            "pdf": str(pdf_path),
            "pdf_id": pdf_id,
            "out_dir": str(out_dir),
            "images_dir": str(images_dir),
            "elements_path": str(elements_path),
            "extracted_images": [str(p) for p in extracted_images],
            "partition_error": error,
        })

    index_path = output_root / "partition_index.json"
    index_path.write_text(json.dumps(index, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\nIndex saved:", index_path.resolve())
    return index

partition_index = partition_folder_pdfs(pdf_paths, OUTPUT_ROOT)

In [ ]:
from transformers import AutoModelForCausalLM
from PIL import Image
import torch

# ----------------------------
# Load Moondream (robust GPU/CPU)
# ----------------------------
def load_moondream2():
    model_id = "vikhyatk/moondream2"
    revision = "2025-06-21"
    if torch.cuda.is_available():
        device_map = {"": "cuda"}
        print("Moondream device: CUDA")
    else:
        device_map = {"": "cpu"}
        print("Moondream device: CPU")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        trust_remote_code=True,
        device_map=device_map,
    )
    return model

moondream = load_moondream2()

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

# ----------------------------
# Summarize extracted images/tables with Moondream
# ----------------------------
def _element_kind(el_type: str) -> str:
    t = (el_type or "").lower()
    if "table" in t:
        return "table"
    if "image" in t or "figure" in t:
        return "image"
    return "image"

def _build_image_kind_map(elements_path: Path) -> dict[str, str]:
    """Map image file basename -> kind (table/image) using elements.jsonl metadata."""
    if not elements_path.exists():
        return {}
    kind_by_basename: dict[str, str] = {}
    with open(elements_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            image_path = (rec.get("image_path") or "").strip()
            if not image_path:
                continue
            el_type = str(rec.get("type") or "")
            kind = _element_kind(el_type)
            base = Path(image_path).name
            if base:
                # If conflicting, prefer 'table' if any element says it's a table
                if base in kind_by_basename and kind_by_basename[base] != kind:
                    if kind == "table":
                        kind_by_basename[base] = "table"
                else:
                    kind_by_basename[base] = kind
    return kind_by_basename

def summarize_image_with_moondream(image_path: Path, kind: str = "image") -> dict:
    img = Image.open(image_path).convert("RGB")
    short_caption = moondream.caption(img, length="short")["caption"]
    normal_caption = moondream.caption(img, length="normal")["caption"]
    if kind == "table":
        prompt = (
            "Ringkas isi tabel ini dalam Bahasa Indonesia. "
            "Sebutkan topik/judul (jika terlihat), kolom penting, dan 3-7 poin informasi utama."
        )
    else:
        prompt = (
            "Ringkas isi gambar ini dalam Bahasa Indonesia. "
            "Berikan 3-7 poin, dan sebutkan teks penting yang terbaca jika ada."
        )
    answer = moondream.query(img, prompt)["answer"]
    return {
        "short_caption": short_caption,
        "normal_caption": normal_caption,
        "summary": answer,
    }

summaries_path = OUTPUT_ROOT / "moondream_summaries.jsonl"
written = 0
ts = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

with open(summaries_path, "w", encoding="utf-8") as out_f:
    for pdf_item in (partition_index or {}).get("pdfs", []):
        pdf = pdf_item.get("pdf")
        pdf_id = pdf_item.get("pdf_id")
        elements_path = Path(pdf_item.get("elements_path") or "")
        kind_map = _build_image_kind_map(elements_path)

        images = [Path(p) for p in (pdf_item.get("extracted_images") or [])]
        if not images:
            continue
        for img_path in images:
            kind = kind_map.get(img_path.name, "image")
            try:
                result = summarize_image_with_moondream(img_path, kind=kind)
                record = {
                    "ts": ts,
                    "pdf": pdf,
                    "pdf_id": pdf_id,
                    "kind": kind,
                    "image_path": str(img_path),
                    **result,
                }
                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                written += 1
            except Exception as e:
                record = {
                    "ts": ts,
                    "pdf": pdf,
                    "pdf_id": pdf_id,
                    "kind": kind,
                    "image_path": str(img_path),
                    "error": repr(e),
                }
                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                written += 1
                print("[WARN] gagal summarize:", img_path, "->", repr(e))

print("Summaries saved:", summaries_path.resolve())
print("Total records:", written)